# Scaling the recommender

🎯 **Goal**: Scale the recommender from a small subset to the full (filtered) dataset, and see how well the current approach holds up.

So far, the recommender has been built and tested on a subset of games to keep things fast and manageable. It works well, but coverage is limited.

Next step: run it on a much larger slice of the data and see what breaks.

**Approach**:

- Rebuild the user–item matrix on the full filtered dataset
- Sense-check size before doing anything expensive
- Try the same similarity approach
- Adapt as needed

### Imports

In [22]:
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"

RAW_PATH = DATA_DIR / "raw"
PROCESSED_PATH = DATA_DIR / "processed"

import pandas as pd

from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors

## Load and filter the data

In [4]:
# Load games metadata
games = pd.read_csv(PROCESSED_PATH / "games.csv")
games.head()

,BGGId,Name,NumUserRatings
0,1,Die Macher,5354
1,2,Dragonmaster,562
2,3,Samurai,15146
3,4,Tal der Könige,340
4,5,Acquire,18655


In [2]:
# Load ratings df
ratings = pd.read_csv(RAW_PATH / "user_ratings.csv")
ratings.head()

,BGGId,Rating,Username
0,213788,8.0,Tonydorrf
1,213788,8.0,tachyon14k
2,213788,8.0,Ungotter
3,213788,8.0,brainlocki3
4,213788,8.0,PPMP


In [6]:
# Drop 63 rows with missing usernames (vs 411k unique users)
ratings = ratings.dropna(subset=["Username"])

# Apply threshold for minimum number of ratings per user and game
min_user_ratings = 10
min_game_ratings = 50

# Filter active users
ratings_per_user = ratings.groupby("Username")["BGGId"].count()

active_users = ratings_per_user[ratings_per_user >= min_user_ratings].index
ratings_filtered = ratings[ratings["Username"].isin(active_users)]

# Filter popular games
ratings_per_game = ratings.groupby("BGGId")["Username"].count()

popular_games = ratings_filtered["BGGId"].value_counts()
popular_games = popular_games[popular_games >= min_game_ratings].index

# Combine
ratings_filtered = ratings_filtered[ratings_filtered["BGGId"].isin(popular_games)]

print("Users:", ratings_filtered["Username"].nunique())
print("Games:", ratings_filtered["BGGId"].nunique())

Users: 224604
Games: 17165


## Feasibility Check

The next step is to test whether the same item-based collaborative filtering approach is still practical on the full filtered dataset. 

To avoid crashing the notebook, I first estimate the size of the matrices involved before attempting the full similarity calculation.

### User-item matrix

In [8]:
# Rebuild on full filtered data
user_item_matrix_full = ratings_filtered.pivot_table(
    index="Username",
    columns="BGGId",
    values="Rating",
    aggfunc="mean"
)

# Check size
n_users, n_games = user_item_matrix_full.shape

print(f"Users: {n_users:,}")
print(f"Games: {n_games:,}")
print(f"Potential similarity matrix size: {n_games * n_games:,}")

/var/folders/y6/tx56lwk52z774x78n87ks5d80000gn/T/ipykernel_84688/540621439.py:2: PerformanceWarning: The following operation may generate 3855327660 cells in the resulting pandas object.
  user_item_matrix_full = ratings_filtered.pivot_table(


Users: 224,604
Games: 17,165
Potential similarity matrix size: 294,637,225


In [9]:
# Memory estimate
n_cells = n_games * n_games
memory_gb = n_cells * 8 / 1e9  # float64

print(f"Estimated similarity matrix size: {memory_gb:.2f} GB")

Estimated similarity matrix size: 2.36 GB


### Item-User matrix

In [10]:
# Fill missing ratings with 0, then transpose so rows = games
item_user_matrix_full = user_item_matrix_full.fillna(0).T

print(item_user_matrix_full.shape)

(17165, 224604)


In [11]:
# Estimate size
n_games_full, n_users_full = item_user_matrix_full.shape
filled_cells = n_games_full * n_users_full
filled_memory_gb = filled_cells * 8 / 1e9  # float64 estimate

print(f"Estimated filled item-user matrix size: {filled_memory_gb:.2f} GB")

Estimated filled item-user matrix size: 30.84 GB


👉 A quick size check shows that converting the full filtered user–item matrix to a dense item–user matrix would require roughly **30GB of memory**. That makes the straightforward dense approach impractical on a typical laptop, even before computing full item–item similarity.

So, while the collaborative filtering logic still holds, scaling this version of the recommender requires a more memory-efficient implementation: **a sparse approach** (storing only the ratings that actually exist).

## Build a sparse matrix

In [17]:
ratings_filtered.columns.tolist()

['BGGId', 'Rating', 'Username']

In [19]:
ratings_sparse_df = ratings_filtered.copy()

# Convert ratings and game IDs into integers
ratings_sparse_df["user_idx"] = ratings_sparse_df["Username"].astype("category").cat.codes
ratings_sparse_df["game_idx"] = ratings_sparse_df["BGGId"].astype("category").cat.codes

# Keep lookup tables
user_lookup = ratings_sparse_df[["Username", "user_idx"]].drop_duplicates()
game_lookup = ratings_sparse_df[["BGGId", "game_idx"]].drop_duplicates()

print(f"Unique users: {user_lookup['user_idx'].nunique():,}")
print(f"Unique games: {game_lookup['game_idx'].nunique():,}")
print(f"Ratings rows: {len(ratings_sparse_df):,}")

Unique users: 224,604
Unique games: 17,165
Ratings rows: 18,194,617


Instead of using a dense pivot table, the full dataset is stored as a sparse matrix. 

This avoids filling missing values with zeros, which would require tens of gigabytes of memory. 

The matrix is first constructed in COO format from the raw ratings data, then converted to CSR format for more efficient computation.

In [20]:
# Build sparse COO matrix
# rows = users, columns = games, values = ratings

user_item_sparse = coo_matrix((
        ratings_sparse_df["Rating"],
        (ratings_sparse_df["user_idx"], ratings_sparse_df["game_idx"])
))

print(user_item_sparse.shape)
print(f"Non-zero entries: {user_item_sparse.nnz:,}")

(224604, 17165)
Non-zero entries: 18,194,617


In [21]:
# Convert to CSR
user_item_sparse = user_item_sparse.tocsr()

print(user_item_sparse.shape)
print(f"Non-zero entries: {user_item_sparse.nnz:,}")

(224604, 17165)
Non-zero entries: 18,162,476


In [23]:
# Transpose to item-user matrix
# rows = games, columns = users

item_user_sparse = user_item_sparse.T

print(item_user_sparse.shape)

(17165, 224604)


🙋‍♀️ Instead of computing a full item–item similarity matrix, **nearest neighbours** are used to retrieve the top similar games for each item. This avoids constructing a large dense matrix, while preserving the core collaborative filtering logic.

In [24]:
# Fit nearest neighbours model
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",   # works well with sparse data
    n_neighbors=51       # 50 + itself
)

knn_model.fit(item_user_sparse)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",51
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [ ]:
# Compute neighbours for all games
distances, indices = knn_model.kneighbors(item_user_sparse)

In [ ]:
# Similarity lookup
item_similarity_dict = {}

for game_idx in range(indices.shape[0]):

    similar_games = indices[game_idx]
    similarities = 1 - distances[game_idx]

    # Skip first game entry (self)
    similar_games = similar_games[1:]
    similarities = similarities[1:]

    item_similarity_dict[game_idx] = list(zip(similar_games, similarities))

## Validate the scaled implementation

In [28]:
# Create reverse lookup
idx_to_bggid = dict(zip(game_lookup["game_idx"], game_lookup["BGGId"]))
bggid_to_idx = dict(zip(game_lookup["BGGId"], game_lookup["game_idx"]))

In [31]:
# Build mappings from games dataset
bggid_to_name = dict(zip(games["BGGId"], games["Name"]))
name_to_bggid = dict(zip(games["Name"], games["BGGId"]))

In [32]:
# Rebuild similar games function with sparse neighbour dict
def get_similar_games_sparse(game_name, top_n=10):
    """
    Return top similar games from the sparse/KNN item neighbour lookup.
    """
    if game_name not in name_to_bggid:
        print(f"{game_name} not found")
        return None

    bggid = name_to_bggid[game_name]

    if bggid not in bggid_to_idx:
        print(f"{game_name} not in sparse similarity model")
        return None

    game_idx = bggid_to_idx[bggid]

    results = []
    for similar_idx, similarity in item_similarity_dict[game_idx][:top_n]:
        similar_bggid = idx_to_bggid[similar_idx]
        similar_name = bggid_to_name.get(similar_bggid, "Unknown")

        results.append({
            "BGGId": similar_bggid,
            "Name": similar_name,
            "Score": similarity
        })

    similar_df = pd.DataFrame(results)

    print(f"Similar to: {game_name}")
    return similar_df[["Name", "Score"]]

In [33]:
# Check results
get_similar_games_sparse("Die Macher", top_n=10)

Similar to: Die Macher


,Name,Score
0,Age of Steam,0.367862
1,Goa,0.361425
2,Taj Mahal,0.353330
3,Imperial,0.348515
4,The Princes of Florence,0.346476
5,Amun-Re,0.342779
6,Indonesia,0.340111
7,Ra,0.324469
8,Automobile,0.323529
9,El Grande,0.321900


In [34]:
get_similar_games_sparse("Wingspan", top_n=10)

Similar to: Wingspan


,Name,Score
0,Azul,0.548648
1,Terraforming Mars,0.527264
2,Everdell,0.495564
3,The Quacks of Quedlinburg,0.486356
4,Scythe,0.481579
5,7 Wonders Duel,0.476825
6,Viticulture Essential Edition,0.468134
7,Sagrada,0.432804
8,Codenames,0.428816
9,The Crew: The Quest for Planet Nine,0.422574


In [35]:
# Rebuild recommender function with sparse dict
def recommend_for_user_sparse(
    liked_games,
    user_ratings,
    bggid_to_idx,
    idx_to_bggid,
    bggid_to_name,
    item_similarity_dict,
    similarity_threshold=0.25,
    top_k_similar=50,
    top_n=20
):
    """
    Recommender using sparse/KNN neighbour lookup instead of dense similarity matrix.
    """
    recommendation_scores = {}

    for liked_game_id, user_rating in liked_games.items():
        if liked_game_id not in bggid_to_idx:
            continue

        liked_game_idx = bggid_to_idx[liked_game_id]

        # Pull similar games from the sparse lookup
        similar_games = item_similarity_dict[liked_game_idx][:top_k_similar]

        for candidate_idx, similarity_score in similar_games:
            if similarity_score < similarity_threshold:
                continue

            candidate_bggid = idx_to_bggid[candidate_idx]

            # Skip games the user already rated
            if candidate_bggid in user_ratings.index:
                continue

            weighted_score = similarity_score * user_rating

            if candidate_bggid not in recommendation_scores:
                recommendation_scores[candidate_bggid] = 0

            recommendation_scores[candidate_bggid] += weighted_score

    recommendations_df = pd.DataFrame(
        list(recommendation_scores.items()),
        columns=["BGGId", "Score"]
    )

    recommendations_df = recommendations_df.sort_values("Score", ascending=False)
    recommendations_df["Name"] = recommendations_df["BGGId"].map(bggid_to_name)

    return recommendations_df[["BGGId", "Name", "Score"]].head(top_n)

In [38]:
# ----- Check outputs for a niche user -----

# Select most active user in the subset
user_ratings_counts = ratings_filtered["Username"].value_counts()
niche_user = user_ratings_counts.index[0]

# Get all ratings for this user from the full filtered dataset
niche_ratings_df = ratings_filtered[ratings_filtered["Username"] == niche_user]

# Convert to a Series indexed by BGGId
niche_ratings = (niche_ratings_df.groupby("BGGId")["Rating"].mean())

# Only keep the games they really liked
liked_games_niche = niche_ratings[niche_ratings >= 8].sort_values(ascending=False)

print(f"Number of liked games (rating >= 8): {len(liked_games_niche)}")

print(f"Niche user: {niche_user}")
print(f"Number of liked games (rating >= 8): {len(liked_games_niche)}")

recommendations_df_niche_sparse = recommend_for_user_sparse(
    liked_games=liked_games_niche,
    user_ratings=niche_ratings,
    bggid_to_idx=bggid_to_idx,
    idx_to_bggid=idx_to_bggid,
    bggid_to_name=bggid_to_name,
    item_similarity_dict=item_similarity_dict,
    similarity_threshold=0.25,
    top_k_similar=50,
    top_n=20
)

recommendations_df_niche_sparse

Number of liked games (rating >= 8): 1228
Niche user: oldgoat3769967
Number of liked games (rating >= 8): 1228


,BGGId,Name,Score
29,555,The Princes of Florence,127.621169
30,22345,Yspahan,94.025767
2,276025,Maracaibo,87.352762
66,66362,Glen More,83.987725
39,39351,Automobile,55.994799
72,154809,Nippon,46.980012
4,91,Paths of Glory,45.344115
38,27833,Steam,44.603175
33,8125,Santiago,42.406384
45,31730,Chicago Express,41.314327


In [44]:
liked_games_niche_df = liked_games_niche.sort_values(ascending=False).rename("UserRating").reset_index()
liked_games_niche_df["Name"] = liked_games_niche_df["BGGId"].map(bggid_to_name)

liked_games_niche_df[["Name", "UserRating"]]

,Name,UserRating
0,Rum & Bones: Second Tide,10.0
1,Band of Brothers: Screaming Eagles,10.0
2,Robinson Crusoe: Adventures on the Cursed Island,10.0
3,Ghost Stories,10.0
4,Smartphone Inc.,10.0
...,...,...
1223,Funkoverse Strategy Game: Golden Girls 101,8.0
1224,Carpe Diem,8.0
1225,The Dead Eye,8.0
1226,NEOM,8.0


In [39]:
# ----- Check outputs for a typical user -----
# User with 10-30 ratings

# Count ratings per user in the full filtered dataset
user_counts = ratings_filtered["Username"].value_counts()

# Filter to "typical" range
typical_users = user_counts[(user_counts >= 10) & (user_counts <= 30)]

# Pick the first typical user
typical_user = typical_users.index[0]

# Get all ratings for this user
typical_ratings_df = ratings_filtered[ratings_filtered["Username"] == typical_user]

# Collapse any duplicate user-game ratings safely
typical_ratings = typical_ratings_df.groupby("BGGId")["Rating"].mean()

# Keep only the games they really liked
liked_games_typical = typical_ratings[typical_ratings >= 8].sort_values(ascending=False)

print(f"Typical user: {typical_user}")
print(f"Number of ratings: {len(typical_ratings)}")
print(f"Number of liked games (rating >= 8): {len(liked_games_typical)}")

recommendations_df_typical_sparse = recommend_for_user_sparse(
    liked_games=liked_games_typical,
    user_ratings=typical_ratings,
    bggid_to_idx=bggid_to_idx,
    idx_to_bggid=idx_to_bggid,
    bggid_to_name=bggid_to_name,
    item_similarity_dict=item_similarity_dict,
    similarity_threshold=0.25,
    top_k_similar=50,
    top_n=20
)

recommendations_df_typical_sparse

Typical user: LuckyTiger32
Number of ratings: 30
Number of liked games (rating >= 8): 10


,BGGId,Name,Score
8,84876,The Castles of Burgundy,24.761574
25,157354,Five Tribes,17.695479
7,230802,Azul,17.053200
0,167791,Terraforming Mars,17.036353
9,163412,Patchwork,16.315432
36,34635,Stone Age,15.861398
15,68448,7 Wonders,15.588253
12,178900,Codenames,15.466067
4,124361,Concordia,14.781499
16,70323,King of Tokyo,14.161076


In [43]:
liked_games_typical_df = liked_games_typical.sort_values(ascending=False).rename("UserRating").reset_index()
liked_games_typical_df["Name"] = liked_games_typical_df["BGGId"].map(bggid_to_name)

liked_games_typical_df[["Name", "UserRating"]]

,Name,UserRating
0,Agricola (Revised Edition),9.6
1,The Red Cathedral,9.4
2,Star Realms,9.0
3,Imperial Settlers: Empires of the North,9.0
4,Trajan,8.8
5,Pulsar 2849,8.5
6,Wingspan,8.5
7,Pandemic,8.3
8,Fresco,8.1
9,Dixit,8.0


In [45]:
# ----- Check outputs for a hobbyist user -----

# Count ratings per user in the full filtered dataset
user_counts = ratings_filtered["Username"].value_counts()

# Filter to the same general activity range
candidate_users = user_counts[(user_counts >= 10) & (user_counts <= 30)]

# Find someone with a stronger taste profile (>= 8 liked games),
# but make sure it's not the same person as the typical user
for username in candidate_users.index:
    if username == typical_user:
        continue

    hobbyist_ratings_df = ratings_filtered[ratings_filtered["Username"] == username]
    hobbyist_ratings = hobbyist_ratings_df.groupby("BGGId")["Rating"].mean()
    hobbyist_liked_games = hobbyist_ratings[hobbyist_ratings >= 8].sort_values(ascending=False)

    if len(hobbyist_liked_games) >= 8:
        hobbyist_user = username
        break

print(f"Hobbyist user: {hobbyist_user}")
print(f"Number of ratings: {len(hobbyist_ratings)}")
print(f"Number of liked games (rating >= 8): {len(hobbyist_liked_games)}")

recommendations_df_hobbyist_sparse = recommend_for_user_sparse(
    liked_games=hobbyist_liked_games,
    user_ratings=hobbyist_ratings,
    bggid_to_idx=bggid_to_idx,
    idx_to_bggid=idx_to_bggid,
    bggid_to_name=bggid_to_name,
    item_similarity_dict=item_similarity_dict,
    similarity_threshold=0.25,
    top_k_similar=50,
    top_n=20
)

recommendations_df_hobbyist_sparse

Hobbyist user: makvlad
Number of ratings: 30
Number of liked games (rating >= 8): 10


,BGGId,Name,Score
5,91,Paths of Glory,13.459021
0,26457,Successors (Third Edition),13.067193
9,38996,Washington's War,11.621013
1,21050,Combat Commander: Europe,10.906849
15,1822,Wilderness War,10.370518
19,833,For the People,9.177632
28,36399,The Napoleonic Wars (Second Edition),7.418080
17,1513,The Republic of Rome,6.555080
25,23679,"Warriors of God: The Wars of England & France,...",6.312067
29,41066,Virgin Queen,6.031041


In [46]:
hobbyist_liked_games_df = hobbyist_liked_games.sort_values(ascending=False).rename("UserRating").reset_index()
hobbyist_liked_games_df["Name"] = hobbyist_liked_games_df["BGGId"].map(bggid_to_name)

hobbyist_liked_games_df[["Name", "UserRating"]]

,Name,UserRating
0,Hellenes: Campaigns of the Peloponnesian War,9.0
1,Commands & Colors: Ancients,9.0
2,Fighting Formations: Grossdeutschland Motorize...,9.0
3,Hannibal: Rome vs. Carthage,8.0
4,Sword of Rome,8.0
5,Here I Stand,8.0
6,Unhappy King Charles!,8.0
7,Down in Flames: Aces High,8.0
8,Warriors of Japan: A Country Aflame 1335-1339,8.0
9,Tenkatoitsu,8.0


### Check versus simple baseline

In [49]:
# load original games data
games_og = pd.read_csv(RAW_PATH / "games.csv")

games_og.head()

,BGGId,Name,Description,YearPublished,GameWeight,AvgRating,BayesAvgRating,StdDev,MinPlayers,MaxPlayers,...,Rank:partygames,Rank:childrensgames,Cat:Thematic,Cat:Strategy,Cat:War,Cat:Family,Cat:CGS,Cat:Abstract,Cat:Party,Cat:Childrens
0,1,Die Macher,die macher game seven sequential political rac...,1986,4.3206,7.61428,7.10363,1.57979,3,5,...,21926,21926,0,1,0,0,0,0,0,0
1,2,Dragonmaster,dragonmaster tricktaking card game base old ga...,1981,1.9630,6.64537,5.78447,1.45440,3,4,...,21926,21926,0,1,0,0,0,0,0,0
2,3,Samurai,samurai set medieval japan player compete gain...,1998,2.4859,7.45601,7.23994,1.18227,2,4,...,21926,21926,0,1,0,0,0,0,0,0
3,4,Tal der Könige,triangular box luxurious large block tal der k...,1992,2.6667,6.60006,5.67954,1.23129,2,4,...,21926,21926,0,0,0,0,0,0,0,0
4,5,Acquire,acquire player strategically invest business t...,1964,2.5031,7.33861,7.14189,1.33583,2,6,...,21926,21926,0,1,0,0,0,0,0,0


In [50]:
# Define fields required for baseline
baseline_df = games_og[["BGGId", "Name", "BayesAvgRating", "NumUserRatings"]].copy()

# Keep only games with a reasonable number of ratings
baseline_df = baseline_df[baseline_df["NumUserRatings"] >= 50]

# Sort by strongest overall games
# BayesAvgRating is better than raw average because it is more stable
baseline_df = baseline_df.sort_values(
    ["BayesAvgRating", "NumUserRatings"],
    ascending=[False, False]
)

# Show top baseline recommendations
baseline_df[["Name", "BayesAvgRating", "NumUserRatings"]].head(20)

,Name,BayesAvgRating,NumUserRatings
14509,Gloomhaven,8.51488,47151
13702,Pandemic Legacy: Season 1,8.44451,44614
17329,Brass: Birmingham,8.41573,24849
14059,Terraforming Mars,8.27421,73093
17834,Twilight Imperium: Fourth Edition,8.25955,15736
20687,Gloomhaven: Jaws of the Lion,8.25163,15062
17127,Gaia Project,8.17758,18791
15360,Star Wars: Rebellion,8.17121,25270
15046,Through the Ages: A New Story of Civilization,8.15369,25341
11220,War of the Ring: Second Edition,8.13104,15247


In [52]:
# ----- Niche vs baseline -----

# Top 20 baseline recommendations
baseline_top_20 = baseline_df[["Name", "BayesAvgRating", "NumUserRatings"]].head(20).copy()

# Top 20 personalised recommendations
niche_personalised_top_20 = recommendations_df_niche_sparse[["Name", "Score"]].head(20).copy()

# Check overlap in game names
baseline_names = set(baseline_top_20["Name"])
niche_personalised_names = set(niche_personalised_top_20["Name"])

niche_overlap_names = baseline_names.intersection(niche_personalised_names)

print("BASELINE")
display(baseline_top_20)

print("PERSONALISED - NICHE USER")
display(niche_personalised_top_20)

niche_overlap_pct = len(niche_overlap_names) / len(niche_personalised_top_20) * 100

print(f"Overlap with baseline: {niche_overlap_pct:.1f}%")

BASELINE


,Name,BayesAvgRating,NumUserRatings
14509,Gloomhaven,8.51488,47151
13702,Pandemic Legacy: Season 1,8.44451,44614
17329,Brass: Birmingham,8.41573,24849
14059,Terraforming Mars,8.27421,73093
17834,Twilight Imperium: Fourth Edition,8.25955,15736
20687,Gloomhaven: Jaws of the Lion,8.25163,15062
17127,Gaia Project,8.17758,18791
15360,Star Wars: Rebellion,8.17121,25270
15046,Through the Ages: A New Story of Civilization,8.15369,25341
11220,War of the Ring: Second Edition,8.13104,15247


PERSONALISED - NICHE USER


,Name,Score
29,The Princes of Florence,127.621169
30,Yspahan,94.025767
2,Maracaibo,87.352762
66,Glen More,83.987725
39,Automobile,55.994799
72,Nippon,46.980012
4,Paths of Glory,45.344115
38,Steam,44.603175
33,Santiago,42.406384
45,Chicago Express,41.314327


Overlap with baseline: 0.0%


In [53]:
# ----- Hobbyist vs baseline -----

# Top 20 personalised recommendations
hobbyist_personalised_top_20 = recommendations_df_hobbyist_sparse[["Name", "Score"]].head(20).copy()

# Check overlap in game names
baseline_names = set(baseline_top_20["Name"])
hobbyist_personalised_names = set(hobbyist_personalised_top_20["Name"])

hobbyist_overlap_names = baseline_names.intersection(hobbyist_personalised_names)

print("BASELINE")
display(baseline_top_20)

print("PERSONALISED - HOBBYIST USER")
display(hobbyist_personalised_top_20)

hobbyist_overlap_pct = len(hobbyist_overlap_names) / len(hobbyist_personalised_top_20) * 100

print(f"Overlap with baseline: {hobbyist_overlap_pct:.1f}%")

BASELINE


,Name,BayesAvgRating,NumUserRatings
14509,Gloomhaven,8.51488,47151
13702,Pandemic Legacy: Season 1,8.44451,44614
17329,Brass: Birmingham,8.41573,24849
14059,Terraforming Mars,8.27421,73093
17834,Twilight Imperium: Fourth Edition,8.25955,15736
20687,Gloomhaven: Jaws of the Lion,8.25163,15062
17127,Gaia Project,8.17758,18791
15360,Star Wars: Rebellion,8.17121,25270
15046,Through the Ages: A New Story of Civilization,8.15369,25341
11220,War of the Ring: Second Edition,8.13104,15247


PERSONALISED - HOBBYIST USER


,Name,Score
5,Paths of Glory,13.459021
0,Successors (Third Edition),13.067193
9,Washington's War,11.621013
1,Combat Commander: Europe,10.906849
15,Wilderness War,10.370518
19,For the People,9.177632
28,The Napoleonic Wars (Second Edition),7.418080
17,The Republic of Rome,6.555080
25,"Warriors of God: The Wars of England & France,...",6.312067
29,Virgin Queen,6.031041


Overlap with baseline: 0.0%


In [54]:
# ----- Typical vs baseline -----

# Top 20 personalised recommendations
typical_personalised_top_20 = recommendations_df_typical_sparse[["Name", "Score"]].head(20).copy()

# Check overlap in game names
baseline_names = set(baseline_top_20["Name"])
typical_personalised_names = set(typical_personalised_top_20["Name"])

typical_overlap_names = baseline_names.intersection(typical_personalised_names)

print("BASELINE")
display(baseline_top_20)

print("PERSONALISED - TYPICAL USER")
display(typical_personalised_top_20)

typical_overlap_pct = len(typical_overlap_names) / len(typical_personalised_top_20) * 100

print(f"Overlap with baseline: {typical_overlap_pct:.1f}%")

BASELINE


,Name,BayesAvgRating,NumUserRatings
14509,Gloomhaven,8.51488,47151
13702,Pandemic Legacy: Season 1,8.44451,44614
17329,Brass: Birmingham,8.41573,24849
14059,Terraforming Mars,8.27421,73093
17834,Twilight Imperium: Fourth Edition,8.25955,15736
20687,Gloomhaven: Jaws of the Lion,8.25163,15062
17127,Gaia Project,8.17758,18791
15360,Star Wars: Rebellion,8.17121,25270
15046,Through the Ages: A New Story of Civilization,8.15369,25341
11220,War of the Ring: Second Edition,8.13104,15247


PERSONALISED - TYPICAL USER


,Name,Score
8,The Castles of Burgundy,24.761574
25,Five Tribes,17.695479
7,Azul,17.053200
0,Terraforming Mars,17.036353
9,Patchwork,16.315432
36,Stone Age,15.861398
15,7 Wonders,15.588253
12,Codenames,15.466067
4,Concordia,14.781499
16,King of Tokyo,14.161076


Overlap with baseline: 25.0%


## ⭐️ Conclusion

Comparing against a popularity-based baseline shows a clear shift in behaviour across user types. The typical user retains some overlap with widely liked games, while both the hobbyist and niche users receive fully personalised recommendations with no overlap.

Importantly, these recommendations remain coherent and aligned with each user’s preferences, suggesting the model is not simply avoiding popular games, but is instead capturing meaningful structure in user taste.

Overall, this indicates that the chosen parameters strike a good balance between filtering noise and retaining enough signal to generate relevant recommendations.

## ⚒️ Save artefacts for app

In [ ]:
# Convert item_similarity_dict into dataframe
rows = []

for game_idx, neighbours in item_similarity_dict.items():
    source_bggid = idx_to_bggid[game_idx]

    for neighbour_idx, score in neighbours:
        neighbour_bggid = idx_to_bggid[neighbour_idx]

        rows.append({
            "BGGId": source_bggid,
            "SimilarBGGId": neighbour_bggid,
            "Score": score
        })

item_neighbours_df = pd.DataFrame(rows)

item_neighbours_df.head()

,BGGId,SimilarBGGId,Score
0,1,4098,0.367862
1,1,9216,0.361425
2,1,475,0.353330
3,1,24181,0.348515
4,1,555,0.346476


In [56]:
item_neighbours_df.to_parquet("../data/processed/item_neighbours.parquet", index=False)

item_neighbours_df.head(1000).to_csv("../data/processed/item_neighbours_sample.csv", index=False)